# Functional Ensemble Detection — Leiden on XGBoost Δ pR²

Identifies co-active neuronal ensembles using Δ pR² from the pairwise XGBoost assay
as the similarity matrix, following the Leiden CPM approach of Bhatt et al. (2026).

**Session:** M29 D23, VR and OF1

**Key difference from Pearson approach:**  
Δ pR² = pR²(null + covariate cell) − pR²(null alone).  
This isolates coordination above chance, with no behavioural variance removed,
so it reflects genuine spike-train co-activity rather than shared tuning.

**Requires:** pairwise assay CSVs produced by `xgboost_pairwise_assay.py`  
Expected path: `{data_path}/xgboost_pairwise_M29_D23_h{hl}_{session}_{start}_{end}.csv`

In [ ]:
import numpy as np
import time
import pandas as pd
import pynapple as nap
import glob
import os
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter
from spatial_manifolds.detect_grids import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

plt.rcParams['font.family'] = 'Arial'

mouse        = 29
day          = 23
source_path  = '/Users/harryclark/Downloads/COHORT12/'
data_path    = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/xgboost_pairwise/'
fig_path     = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_playground/'

HISTORY_LENGTH = 1000   # which history length to load / compute

# ── Baseline selection ────────────────────────────────────────────────────────
# Δ pR² = pR²(BASELINE + covariate cell) - pR²(BASELINE alone)
#
# VR baselines available:
#   'null'          — zeros (above chance)
#   'pos'           — track position
#   'speed'         — running speed
#   'lfp'           — theta LFP
#   'pos_speed'     — position + speed
#   'pos_lfp'       — position + LFP
#   'speed_lfp'     — speed + LFP
#   'pos_speed_lfp' — position + speed + LFP  (most conservative)
#
# Additional OF1 baselines:
#   'hd'                      — head direction
#   'hing'                    — head-in-goal angle
#   'pos_hd'                  — position + HD
#   'pos_hing'                — position + Hing
#   'pos_speed_hd'            — position + speed + HD
#   'pos_speed_hing'          — position + speed + Hing
#   'pos_hd_hing'             — position + HD + Hing
#   'pos_speed_hd_hing'       — position + speed + HD + Hing
#   'pos_speed_hd_hing_lfp'  — all (most conservative for OF)
#
# 'null'         → raw co-activity above chance (no behaviour removed)
# 'pos_speed_lfp' / 'pos_speed_hd_hing_lfp' → coordination above all behaviour
BASELINE_VR = 'null'   # baseline for VR  (set 'null' for raw)
BASELINE_OF = 'null'  # baseline for OF1 (set 'null' for raw)

COL_GC    = '#c04744'
COL_NGS   = '#3171ae'
COL_OTHER = '#aaaaaa'

cell_class = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
sess_cells = cell_class[
    (cell_class['mouse'] == mouse) & (cell_class['day'] == day)
].copy()
sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
print(f'M{mouse} D{day}: {len(sess_cells)} classified cells')
print(f'VR  baseline: {BASELINE_VR}')
print(f'OF1 baseline: {BASELINE_OF}')

## 1. Load session data

In [ ]:
_t0 = time.time()
print('Loading VR...')
tcs_vr, tcs_time_vr, _, last_ephys_bin_vr, beh_vr, clusters_vr = compute_vr_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path)
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')
print(f'  VR loaded in {time.time()-_t0:.1f}s  |  {len(tcs_time_vr)} cells  |  {last_ephys_bin_vr * bs / 1000:.0f} s session')

_t1 = time.time()
print('Loading OF1...')
tcs_of, tcs_time_of, beh_of, clusters_of, ep_of = compute_of_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path, session='OF1')
print(f'  OF1 loaded in {time.time()-_t1:.1f}s  |  {len(tcs_time_of)} cells')
print(f'Session data total: {time.time()-_t0:.1f}s')


## 2. Load or compute pairwise XGBoost Δ pR²

In [ ]:
def load_pairwise_results(mouse, day, history_length, session_type, data_path):
    pattern = os.path.join(
        data_path,
        f'xgboost_pairwise_M{mouse}_D{day}_h{history_length}_{session_type}_*.csv'
    )
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f'  Loaded {len(files)} files → {len(df):,} rows')
    return df


def get_baseline_x(baseline, session_type, beh, ep, T, lfp_trace=None):
    """
    Build the behavioral covariate array for a given baseline name.
    lfp_trace: pre-loaded session-level LFP array (T,); zeros used if None.
    Returns np.ndarray of shape (T, n_covariates).
    """
    def _bin(key):
        arr = np.array(beh[key].bin_average(bin_size=time_bs, time_units='ms', ep=ep))
        if np.any(np.isnan(arr)):
            arr = pd.Series(arr).ffill().bfill().values
        return arr[:T]

    if baseline == 'null':
        return np.zeros((T, 1))

    if session_type == 'VR':
        dt  = _bin('travel') - ((beh['trial_number'][0] - 1) * tl)
        pos = (dt % tl)[:T]
        spd = _bin('S')
        if lfp_trace is not None:
            lfp = np.zeros(T)
            n   = min(len(lfp_trace), T)
            lfp[:n] = lfp_trace[:n]
        else:
            lfp = np.zeros(T)
        mapping = {
            'pos':           pos[:, None],
            'speed':         spd[:, None],
            'lfp':           lfp[:, None],
            'pos_speed':     np.column_stack([pos, spd]),
            'pos_lfp':       np.column_stack([pos, lfp]),
            'speed_lfp':     np.column_stack([spd, lfp]),
            'pos_speed_lfp': np.column_stack([pos, spd, lfp]),
        }
    else:  # OF1
        px   = _bin('P_x');  py  = _bin('P_y')
        spd  = _bin('S');    hd  = _bin('H');  hing = _bin('Hing')
        if lfp_trace is not None:
            lfp = np.zeros(T)
            n   = min(len(lfp_trace), T)
            lfp[:n] = lfp_trace[:n]
        else:
            lfp = np.zeros(T)
        mapping = {
            'pos':                    np.column_stack([px, py]),
            'speed':                  spd[:, None],
            'hd':                     hd[:, None],
            'hing':                   hing[:, None],
            'lfp':                    lfp[:, None],
            'pos_speed':              np.column_stack([px, py, spd]),
            'pos_hd':                 np.column_stack([px, py, hd]),
            'pos_hing':               np.column_stack([px, py, hing]),
            'pos_lfp':                np.column_stack([px, py, lfp]),
            'pos_speed_hd':           np.column_stack([px, py, spd, hd]),
            'pos_speed_hing':         np.column_stack([px, py, spd, hing]),
            'pos_speed_lfp':          np.column_stack([px, py, spd, lfp]),
            'pos_hd_hing':            np.column_stack([px, py, hd, hing]),
            'pos_speed_hd_hing':      np.column_stack([px, py, spd, hd, hing]),
            'pos_speed_hd_hing_lfp':  np.column_stack([px, py, spd, hd, hing, lfp]),
        }

    if baseline not in mapping:
        raise ValueError(f'Unknown baseline "{baseline}" for {session_type}.\n'
                         f'Valid options: {list(mapping.keys())}')
    return mapping[baseline]


def compute_pairwise_inline(tcs_time, beh, ep, mouse, day, session_type,
                             baseline, history_length=200, data_path=None):
    """
    Compute pairwise pR² in-notebook using the specified baseline.
    Saves results to data_path so subsequent runs load from file.
    """
    from spatial_manifolds.mlencoding import MLencoding
    nfilters = int(history_length / time_bs)
    xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                     window=time_bs, n_filters=nfilters, max_time=history_length)

    all_ids  = sorted(tcs_time.keys())
    T        = min(len(np.array(tcs_time[cid])) for cid in all_ids)
    spike_mat = {cid: np.array(tcs_time[cid])[:T] for cid in all_ids}

    n_total   = len(all_ids)
    n_pairs   = n_total * (n_total - 1)
    rows      = []
    _t_start  = time.time()
    print(f'  Baseline: "{baseline}"  |  {n_total} cells  |  {n_pairs} pairs to fit')
    print(f'  Est. time at ~5s/fit: {n_pairs * 5 / 3600:.1f} h')


    for ti, target_id in enumerate(all_ids):
        y = spike_mat[target_id]

        # Load LFP for this cell's closest electrode channel.
        # Falls back to zeros if the LFP file or channel entry is missing.
        try:
            lfp_cell = np.array(get_theta_trace(
                mouse=mouse, day=day, cluster_id=target_id,
                time_bs=50, resample_bs=time_bs,
                session_type=session_type, source_path=source_path,
            ))
        except Exception as _lfp_err:
            print(f'    [LFP fallback] cell {target_id}: {_lfp_err}')
            lfp_cell = np.zeros(T)

        x_bl = get_baseline_x(baseline, session_type, beh, ep, T,
                              lfp_trace=lfp_cell)

        # Baseline-only fit
        _, pr2_bl = xgb.fit_cv(x_bl, y, verbose=0, continuous_folds=True)
        rows.append(dict(
            mouse=mouse, day=day, session_type=session_type,
            target_cluster_id=int(target_id), target_type='unknown',
            covariate_cluster_id=-1, covariate_type='none',
            baseline=baseline, pR2_cv=float(np.nanmean(pr2_bl)),
        ))

        # Pairwise: baseline + each covariate cell
        for cov_id in all_ids:
            if cov_id == target_id:
                continue
            x = np.column_stack([x_bl, spike_mat[cov_id]])
            _, pr2_cv = xgb.fit_cv(x, y, verbose=0, continuous_folds=True)
            rows.append(dict(
                mouse=mouse, day=day, session_type=session_type,
                target_cluster_id=int(target_id), target_type='unknown',
                covariate_cluster_id=int(cov_id), covariate_type='unknown',
                baseline=baseline, pR2_cv=float(np.nanmean(pr2_cv)),
            ))

        _elapsed = time.time() - _t_start
        _rate    = _elapsed / (ti + 1)
        _eta     = _rate * (n_total - ti - 1)
        print(f'  {session_type} [{baseline}]: {ti+1}/{n_total}  '
              f'elapsed={_elapsed:.0f}s  ETA={_eta:.0f}s', end='\r')

    print(f'\n  Done in {time.time()-_t_start:.1f}s')
    df = pd.DataFrame(rows)

    if data_path is not None:
        os.makedirs(data_path, exist_ok=True)
        out = os.path.join(
            data_path,
            f'xgboost_pairwise_M{mouse}_D{day}_h{history_length}_{session_type}_0_{n_total}.csv')
        df.to_csv(out, index=False)
        print(f'  Saved → {out}')
    return df


# ── Load or compute ────────────────────────────────────────────────────────────
# Build epoch intervals needed by get_baseline_x
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')

_t_pw = time.time()
print(f'\n=== VR  [baseline: {BASELINE_VR}] ===')
pw_vr = load_pairwise_results(mouse, day, HISTORY_LENGTH, 'VR', data_path)
if pw_vr is None or BASELINE_VR not in pw_vr['baseline'].values:
    print('  Results not found / baseline missing — computing inline...')
    pw_vr = compute_pairwise_inline(
        tcs_time_vr, beh_vr, ep_vr, mouse, day, 'VR',
        baseline=BASELINE_VR, history_length=HISTORY_LENGTH,
        data_path=data_path)

print(f'\n=== OF1 [baseline: {BASELINE_OF}] ===')
pw_of = load_pairwise_results(mouse, day, HISTORY_LENGTH, 'OF1', data_path)
if pw_of is None or BASELINE_OF not in pw_of['baseline'].values:
    print('  Results not found / baseline missing — computing inline...')
    pw_of = compute_pairwise_inline(
        tcs_time_of, beh_of, ep_of, mouse, day, 'OF1',
        baseline=BASELINE_OF, history_length=HISTORY_LENGTH,
        data_path=data_path)

print(f'VR  pairwise ready in {time.time()-_t_pw:.1f}s  |  {len(pw_vr):,} rows')
print(f'OF1 pairwise ready  |  {len(pw_of):,} rows')
print(f'Baselines VR: {pw_vr["baseline"].unique()}')
print(f'Baselines OF1: {pw_of["baseline"].unique()}')
pw_vr.head()

## 3. Build Δ pR² matrix

In [ ]:
def build_delta_pr2_matrix(pw_df, baseline):
    """
    Build an N×N directed Δ pR² matrix where entry [i, j] =
    pR²(baseline + cell_i predicting cell_j) - pR²(baseline alone predicting cell_j).

    Baseline rows have covariate_cluster_id == -1.
    Cell rows have covariate_cluster_id >= 0.
    """
    # ── Baseline pR² (no covariate cell) ─────────────────────────────────────
    bl = (
        pw_df[(pw_df['baseline'] == baseline) & (pw_df['covariate_cluster_id'] == -1)]
        [['target_cluster_id', 'pR2_cv']]
        .rename(columns={'pR2_cv': 'pR2_baseline'})
        .drop_duplicates('target_cluster_id')
    )

    # ── Cell pR² (with covariate) ─────────────────────────────────────────────
    cv = (
        pw_df[(pw_df['baseline'] == baseline) & (pw_df['covariate_cluster_id'] >= 0)]
        [['target_cluster_id', 'covariate_cluster_id', 'pR2_cv']]
        .copy()
    )

    # ── Merge and compute Δ pR² ───────────────────────────────────────────────
    merged = cv.merge(bl, on='target_cluster_id', how='inner')
    merged['delta_pr2'] = merged['pR2_cv'] - merged['pR2_baseline']

    # ── Build N×N matrix ──────────────────────────────────────────────────────
    all_ids = sorted(set(merged['target_cluster_id']) | set(merged['covariate_cluster_id']))
    id_to_idx = {cid: i for i, cid in enumerate(all_ids)}
    N = len(all_ids)
    mat = np.full((N, N), np.nan)

    for _, row in merged.iterrows():
        i = id_to_idx[int(row['covariate_cluster_id'])]   # covariate = 'predictor'
        j = id_to_idx[int(row['target_cluster_id'])]      # target    = 'predicted'
        mat[i, j] = row['delta_pr2']

    np.fill_diagonal(mat, 0)

    # ── Symmetrise by averaging both directions ───────────────────────────────
    sym = np.nanmean(np.stack([mat, mat.T], axis=0), axis=0)
    sym = np.nan_to_num(sym, nan=0.0)

    return sym, all_ids, mat  # sym = undirected, mat = directed


_t_mat = time.time()
print('Building VR Δ pR² matrix...')
dpr2_vr_sym, ids_vr, dpr2_vr_dir = build_delta_pr2_matrix(pw_vr, baseline=BASELINE_VR)

print('Building OF1 Δ pR² matrix...')
dpr2_of_sym, ids_of, dpr2_of_dir = build_delta_pr2_matrix(pw_of, baseline=BASELINE_OF)

print(f'Matrices built in {time.time()-_t_mat:.1f}s')
print(f'VR:  {dpr2_vr_sym.shape}  Δ pR² range [{dpr2_vr_sym.min():.4f}, {dpr2_vr_sym.max():.4f}]')
print(f'OF1: {dpr2_of_sym.shape}  Δ pR² range [{dpr2_of_sym.min():.4f}, {dpr2_of_sym.max():.4f}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, mat, title in zip(axes,
                           [dpr2_vr_sym, dpr2_of_sym],
                           ['VR', 'OF1']):
    vmax = np.nanpercentile(np.abs(mat), 98)
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.04).set_label('Δ pR²', fontsize=8)
    ax.set_title(f'Symmetrised Δ pR² matrix — {title}', fontsize=10)
    ax.set_xlabel('Covariate cell index')
    ax.set_ylabel('Target cell index')
plt.tight_layout()
plt.savefig(fig_path + f'dpr2_matrix_raw_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Directional asymmetry: how much does A predict B vs B predict A?
# Positive = A→B stronger than B→A
for session, mat_dir, ids in [('VR', dpr2_vr_dir, ids_vr),
                               ('OF1', dpr2_of_dir, ids_of)]:
    asym = mat_dir - mat_dir.T   # positive = row cell drives col cell more
    mean_asym = np.nanmean(np.abs(asym[~np.isnan(asym)]))
    print(f'{session}  mean |asymmetry| = {mean_asym:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, mat_dir, title in zip(axes,
                               [dpr2_vr_dir, dpr2_of_dir],
                               ['VR', 'OF1']):
    asym = mat_dir - mat_dir.T
    vmax = np.nanpercentile(np.abs(asym[~np.isnan(asym)]), 98)
    im = ax.imshow(np.nan_to_num(asym), cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.04).set_label('Δ pR² (i→j) − (j→i)', fontsize=8)
    ax.set_title(f'Directional asymmetry — {title}', fontsize=10)
    ax.set_xlabel('Target cell index')
    ax.set_ylabel('Covariate (driver) cell index')
plt.tight_layout()
plt.savefig(fig_path + f'dpr2_asymmetry_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 4. Leiden community detection (CPM on Δ pR²)

In [ ]:
def run_leiden(sym_mat, n_search=500, res_min=0.5, res_max=1.75,
               n_iter_search=10, n_iter_final=1000, seed=42):
    """
    Leiden CPM with resolution parameter search.
    Uses all Δ pR² values (positive + negative) as edge weights.
    """
    n = sym_mat.shape[0]
    rows, cols = np.triu_indices(n, k=1)
    weights    = sym_mat[rows, cols].tolist()

    g = ig.Graph(n=n,
                 edges=list(zip(rows.tolist(), cols.tolist())),
                 directed=False,
                 edge_attrs={'weight': weights})

    resolutions = np.linspace(res_min, res_max, n_search)
    best_mod    = -np.inf
    best_res    = resolutions[0]
    mod_curve   = []
    _t_res = time.time()
    print(f'  Searching {n_search} resolution values...')

    for res in resolutions:
        part = leidenalg.find_partition(
            g, leidenalg.CPMVertexPartition,
            weights='weight', resolution_parameter=res,
            n_iterations=n_iter_search, seed=seed)
        mod_curve.append(part.modularity)
        if part.modularity > best_mod:
            best_mod = part.modularity
            best_res = res

    print(f'  Resolution search done in {time.time()-_t_res:.1f}s  |  best γ={best_res:.3f}  Q={best_mod:.3f}')
    _t_final = time.time()
    print(f'  Running final Leiden ({n_iter_final} iterations)...')
    part = leidenalg.find_partition(
        g, leidenalg.CPMVertexPartition,
        weights='weight', resolution_parameter=best_res,
        n_iterations=n_iter_final, seed=seed)
    print(f'  Final run done in {time.time()-_t_final:.1f}s')

    labels = np.array(part.membership)
    unique, counts = np.unique(labels, return_counts=True)
    for u, c in zip(unique, counts):
        if c < 2:
            labels[labels == u] = -1

    return labels, best_res, best_mod, resolutions, np.array(mod_curve)


_t_leiden = time.time()
print('Running Leiden (VR)...')
labels_vr, res_vr, mod_vr, res_c_vr, mod_c_vr = run_leiden(dpr2_vr_sym)

print('Running Leiden (OF1)...')
labels_of, res_of, mod_of, res_c_of, mod_c_of = run_leiden(dpr2_of_sym)

n_ens_vr = len(np.unique(labels_vr[labels_vr >= 0]))
n_ens_of = len(np.unique(labels_of[labels_of >= 0]))
print(f'\nVR:  {n_ens_vr} ensembles  (γ={res_vr:.3f}, Q={mod_vr:.3f})')
print(f'OF1: {n_ens_of} ensembles  (γ={res_of:.3f}, Q={mod_of:.3f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, res_c, mod_c, best_r, title in zip(
        axes,
        [res_c_vr, res_c_of], [mod_c_vr, mod_c_of],
        [res_vr, res_of], ['VR', 'OF1']):
    ax.plot(res_c, mod_c, lw=1, color='#3171ae')
    ax.axvline(best_r, color='#c04744', lw=1.5, ls='--', label=f'γ={best_r:.3f}')
    ax.set_xlabel('Resolution (γ)', fontsize=9)
    ax.set_ylabel('Modularity', fontsize=9)
    ax.set_title(f'Resolution search — {title}', fontsize=9)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_path + f'leiden_resolution_search_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 5. Ensemble membership

In [ ]:
def make_member_df(ids, labels, sess_cells):
    cols = ['cluster_id', 'cell_type', 'probe_x', 'probe_y',
            'SC_x', 'SC_y', 'SC_z', 'brain_region']
    available = [c for c in cols if c in sess_cells.columns]
    df = pd.DataFrame({'cluster_id': ids, 'ensemble': labels})
    df = df.merge(sess_cells[available], on='cluster_id', how='left')
    df['SC_x_abs'] = pd.to_numeric(df['SC_x'], errors='coerce').abs()
    df['medlat']   = np.where(df['SC_x_abs'] < 3400, 'medial', 'lateral')
    return df


df_vr = make_member_df(ids_vr, labels_vr, sess_cells)
df_of = make_member_df(ids_of, labels_of, sess_cells)

for session, df, n_ens in [('VR', df_vr, n_ens_vr), ('OF1', df_of, n_ens_of)]:
    print(f'\n── {session} ({n_ens} ensembles) ──')
    for ens in sorted(df[df['ensemble'] >= 0]['ensemble'].unique()):
        sub   = df[df['ensemble'] == ens]
        types = sub['cell_type'].value_counts().to_dict()
        ml    = sub['medlat'].value_counts().to_dict()
        print(f'  E{ens:2d}: n={len(sub):3d}  types={types}  ML={ml}')

## 6. Main figure

In [ ]:
ens_palette = plt.cm.tab20.colors
def ens_color(label): return ens_palette[label % len(ens_palette)] if label >= 0 else '#cccccc'

def sort_by_ensemble(mat, labels):
    order = np.argsort(labels)
    return mat[np.ix_(order, order)], labels[order]

dpr2_vr_s, lbl_vr_s = sort_by_ensemble(dpr2_vr_sym, labels_vr)
dpr2_of_s, lbl_of_s = sort_by_ensemble(dpr2_of_sym, labels_of)


def plot_matrix_sorted(ax, mat_s, lbl_s, title):
    vmax = np.nanpercentile(np.abs(mat_s), 98)
    im = ax.imshow(mat_s, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.03).set_label('Δ pR²', fontsize=7)
    for b in np.where(np.diff(lbl_s) != 0)[0] + 0.5:
        ax.axhline(b, color='black', lw=0.7); ax.axvline(b, color='black', lw=0.7)
    ax.set_title(title, fontsize=9, fontweight='bold', loc='left')
    ax.set_xlabel('Cell (sorted)', fontsize=8); ax.set_ylabel('Cell (sorted)', fontsize=8)
    ax.tick_params(labelsize=7)


def plot_probe(ax, df, title):
    ax.scatter(df[df['ensemble'] < 0]['probe_x'],
               df[df['ensemble'] < 0]['probe_y'],
               c='#cccccc', s=14, zorder=1)
    for ens in sorted(df[df['ensemble'] >= 0]['ensemble'].unique()):
        sub = df[df['ensemble'] == ens]
        ax.scatter(sub['probe_x'], sub['probe_y'],
                   color=ens_color(ens), s=24, zorder=2,
                   edgecolors='k', linewidths=0.4, label=f'E{ens}')
    ax.legend(fontsize=6, frameon=False, ncol=2)
    ax.set_xlabel('Probe x (µm)', fontsize=8); ax.set_ylabel('Probe y (µm)', fontsize=8)
    ax.set_title(title, fontsize=9, fontweight='bold', loc='left')
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(labelsize=7)


def plot_ml_dv(ax, df, title):
    df_v = df[df['ensemble'] >= 0].copy()
    for ens in sorted(df_v['ensemble'].unique()):
        sub = df_v[df_v['ensemble'] == ens]
        ax.scatter(sub['SC_x_abs'], sub['SC_y'],
                   color=ens_color(ens), s=24, alpha=0.85,
                   edgecolors='k', linewidths=0.3)
    ax.axvline(3400, color='black', lw=1.2, ls='--', alpha=0.7)
    ax.set_xlabel('|SC_x| ML (µm)', fontsize=8)
    ax.set_ylabel('SC_y DV (µm)', fontsize=8)
    ax.set_title(title, fontsize=9, fontweight='bold', loc='left')
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_yaxis(); ax.tick_params(labelsize=7)


def plot_ens_mean_dpr2(ax, sym_mat, labels, ids, title):
    """Within-ensemble mean Δ pR² vs cross-ensemble, per ensemble."""
    ensembles = sorted(set(labels[labels >= 0]))
    within, cross = [], []
    for ens in ensembles:
        idx_in  = np.where(labels == ens)[0]
        idx_out = np.where(labels != ens)[0]
        w = sym_mat[np.ix_(idx_in, idx_in)]
        c = sym_mat[np.ix_(idx_in, idx_out)]
        within.append(w[w != 0].mean() if w[w != 0].size else 0)
        cross.append(c.mean())
    x = np.arange(len(ensembles))
    w = 0.35
    ax.bar(x - w/2, within, width=w, color='#3171ae', alpha=0.8, label='Within ensemble')
    ax.bar(x + w/2, cross,  width=w, color='#aaaaaa', alpha=0.8, label='Cross-ensemble')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels([f'E{e}' for e in ensembles], fontsize=7)
    ax.set_ylabel('Mean Δ pR²', fontsize=8)
    ax.set_title(title, fontsize=9, fontweight='bold', loc='left')
    ax.legend(fontsize=7, frameon=False)
    ax.spines[['top', 'right']].set_visible(False); ax.tick_params(labelsize=7)


# ── Build figure ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 4, figure=fig,
                        hspace=0.45, wspace=0.40,
                        left=0.05, right=0.98, top=0.93, bottom=0.08)

for row_i, (session, mat_s, lbl_s, df, sym_mat, ids, n_ens) in enumerate([
    ('VR',  dpr2_vr_s, lbl_vr_s, df_vr, dpr2_vr_sym, ids_vr, n_ens_vr),
    ('OF1', dpr2_of_s, lbl_of_s, df_of, dpr2_of_sym, ids_of, n_ens_of),
]):
    ax0 = fig.add_subplot(gs[row_i, 0])
    ax1 = fig.add_subplot(gs[row_i, 1])
    ax2 = fig.add_subplot(gs[row_i, 2])
    ax3 = fig.add_subplot(gs[row_i, 3])

    plot_matrix_sorted(ax0, mat_s, lbl_s,
                       f'Sorted Δ pR² — {session}\n({n_ens} ensembles)')
    plot_probe(ax1, df, f'Probe anatomy ({session})')
    plot_ml_dv(ax2, df, f'ML × DV anatomy ({session})')
    plot_ens_mean_dpr2(ax3, sym_mat,
                       np.array([lbl_s[lbl_s == lbl_s[i]][0]
                                 if lbl_s[i] >= 0 else -1
                                 for i in range(len(lbl_s))]),
                       ids, f'Within vs cross-ensemble Δ pR² ({session})')

plt.suptitle(f'Functional ensembles from XGBoost Δ pR² — M{mouse} D{day}  (Leiden CPM, h={HISTORY_LENGTH}ms)',
             fontsize=12, fontweight='bold')
plt.savefig(fig_path + f'leiden_xgboost_M{mouse}D{day}.pdf',
            bbox_inches='tight', dpi=200)
plt.show()

## 7. VR vs OF1 ensemble overlap (shared cells)

In [ ]:
shared_ids = sorted(set(ids_vr) & set(ids_of))
print(f'{len(shared_ids)} cells present in both VR and OF1')

vr_lbl = np.array([labels_vr[ids_vr.index(c)] for c in shared_ids])
of_lbl = np.array([labels_of[ids_of.index(c)] for c in shared_ids])

vr_ens = sorted(set(vr_lbl[vr_lbl >= 0]))
of_ens = sorted(set(of_lbl[of_lbl >= 0]))

contingency = np.zeros((len(vr_ens), len(of_ens)))
for i, ve in enumerate(vr_ens):
    for j, oe in enumerate(of_ens):
        contingency[i, j] = np.sum((vr_lbl == ve) & (of_lbl == oe))

row_sums = contingency.sum(axis=1, keepdims=True)
contingency_norm = np.where(row_sums > 0, contingency / row_sums, 0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, data, cblabel in zip(
        axes,
        [contingency, contingency_norm],
        ['n cells', 'fraction of VR ensemble']):
    im = ax.imshow(data, cmap='Blues', aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.04).set_label(cblabel, fontsize=8)
    for i in range(len(vr_ens)):
        for j in range(len(of_ens)):
            v = data[i, j]
            if v > 0:
                fmt = f'{v:.0f}' if cblabel == 'n cells' else f'{v:.2f}'
                ax.text(j, i, fmt, ha='center', va='center', fontsize=8,
                        color='white' if v > data.max() * 0.6 else '#333')
    ax.set_xticks(range(len(of_ens)))
    ax.set_xticklabels([f'OF E{e}' for e in of_ens], fontsize=8)
    ax.set_yticks(range(len(vr_ens)))
    ax.set_yticklabels([f'VR E{e}' for e in vr_ens], fontsize=8)
    ax.set_xlabel('OF1 ensemble'); ax.set_ylabel('VR ensemble')
plt.suptitle(f'VR vs OF1 ensemble overlap — M{mouse} D{day}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(fig_path + f'leiden_vr_of_overlap_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 8. Ensemble composition — cell types and ML structure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f'Ensemble composition — M{mouse} D{day}', fontsize=11, fontweight='bold')

for row_i, (session, df) in enumerate([('VR', df_vr), ('OF1', df_of)]):
    df_v = df[df['ensemble'] >= 0].copy()
    ensembles = sorted(df_v['ensemble'].unique())

    # Stacked bar: cell type
    ax = axes[row_i, 0]
    gc_n    = [len(df_v[(df_v['ensemble']==e) & (df_v['cell_type']=='GC')])  for e in ensembles]
    ngs_n   = [len(df_v[(df_v['ensemble']==e) & (df_v['cell_type']=='NG')])  for e in ensembles]
    other_n = [len(df_v[(df_v['ensemble']==e) & (~df_v['cell_type'].isin(['GC','NG']))]) for e in ensembles]
    x = np.arange(len(ensembles))
    ax.bar(x, gc_n,    color=COL_GC,    label='GC',    alpha=0.85)
    ax.bar(x, ngs_n,   color=COL_NGS,   label='NGS',   alpha=0.85, bottom=gc_n)
    ax.bar(x, other_n, color=COL_OTHER, label='Other', alpha=0.85,
           bottom=[g+n for g,n in zip(gc_n, ngs_n)])
    ax.set_xticks(x); ax.set_xticklabels([f'E{e}' for e in ensembles], fontsize=8)
    ax.set_ylabel('n cells', fontsize=9)
    ax.set_title(f'{session} — cell type composition', fontsize=9)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top','right']].set_visible(False)

    # ML position per ensemble (boxplot)
    ax = axes[row_i, 1]
    ml_data = [df_v[df_v['ensemble']==e]['SC_x_abs'].dropna().values for e in ensembles]
    bp = ax.boxplot(ml_data, positions=range(len(ensembles)), widths=0.5,
                    patch_artist=True, medianprops=dict(color='#333333', lw=1.5),
                    whiskerprops=dict(color='#555555'), capprops=dict(color='#555555'),
                    flierprops=dict(marker='', alpha=0))
    for patch, ens in zip(bp['boxes'], ensembles):
        patch.set(facecolor=ens_color(ens), alpha=0.7, edgecolor='#555555')
    ax.axhline(3400, color='black', lw=1.2, ls='--', alpha=0.7, label='3400 µm')
    ax.set_xticks(range(len(ensembles)))
    ax.set_xticklabels([f'E{e}' for e in ensembles], fontsize=8)
    ax.set_ylabel('|SC_x| — ML position (µm)', fontsize=9)
    ax.set_title(f'{session} — ML position per ensemble', fontsize=9)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(fig_path + f'leiden_composition_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 9. Save membership CSVs

In [ ]:
df_vr.to_csv(fig_path + f'leiden_membership_VR_M{mouse}D{day}.csv',  index=False)
df_of.to_csv(fig_path + f'leiden_membership_OF1_M{mouse}D{day}.csv', index=False)
print('Saved membership CSVs to', fig_path)
df_vr[['cluster_id','ensemble','cell_type','medlat']].sort_values('ensemble')